In [ ]:
# pip freeze > requirements.txt (to save the current environment)

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# TODO: Model import goes here


# NBA API Endpoints
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.endpoints import boxscoresummaryv2



'\nWorkflow:\n1. [FRONTEND] Get player name \n2. [DONE] Get player ID from player name\n3. Fetch career stats, gamelog, team gamelog using player ID\n    Stats to fetch:\n    - Career stats (averaged stats) [Need for predicting over/under prop]\n    - Game log (past season games; grab prev season if <25% season completed; grab past 2 seasons games against certain team) [Need for dataset]\n        - Core stats (PTS, REBOUNDS, ASSISTS, STEALS, 3PT MADE, DOUBLE-DOUBLES, TRIPLE-DOUBLES) [For dataset]\n    - Box score summary (injury report) [Need for injury report]\n    \n4. Form dataset from all fetched endpoints\n5. Preprocess the dataset\n6. Train-test split (or any other splitting method)\n7. Train the model + evaluate (fine tune)\n8. Run predictions\n9. Get LLM to summarize the predictions\n'


## Workflow:
1. [⚠️FRONTEND] Get player name 
2. ✅ Get player ID from player name
3. Fetch career stats, gamelog, team gamelog using player ID
    Stats to fetch:
    - ✅ Career stats (averaged stats) [Need for predicting over/under prop]
    - Game log (past season games; grab prev season if <25% season completed; grab past 2 seasons games against certain team) [Need for dataset]
        - Core stats (PTS, REBOUNDS, ASSISTS, STEALS, 3PT MADE, DOUBLE-DOUBLES, TRIPLE-DOUBLES) [For dataset]
    - Box score summary (injury report) [Need for injury report]
    
4. Form dataset from all fetched endpoints
5. Preprocess the dataset
6. Train-test split (or any other splitting method)
7. Train the model + evaluate (fine tune)
8. Run predictions
9. [⚠️LLM] Get LLM to summarize the predictions


In [3]:
# List of all players in the NBA currently (will be used to autofill in search bar)
from nba_api.stats.static.players import get_players
players = pd.DataFrame(get_players())

# Function to get player ID from player name
def get_player_id(player_name):
    active_players = players[players['is_active']== True]

    # Retrieve player ID provided player name
    player_id = active_players[active_players['full_name'] == player_name]['id'].values[0]
    return player_id

test_name = 'Cade Cunningham' # TODO: Get user input from frontend (MUI autocomplete)

player_id = get_player_id(test_name) # Get player ID from player name
print(test_name, player_id) # Print player ID


Cade Cunningham 1630595


In [ ]:
# Fetching data from NBA API endpoints for dataset creation
avg_career_stats = playercareerstats.PlayerCareerStats(player_id=player_id, per_mode36="PerGame").get_data_frames()[3] # Averaged career stats for regular season [Need for predicting prop]
print(avg_career_stats) # Print average career stats

   PLAYER_ID LEAGUE_ID  TEAM_ID  GP  GS   MIN  FGM   FGA  FG_PCT  FG3M  ...  \
0    1630595        00        0   6   6  41.2  9.2  21.5   0.426   0.8  ...   

   FT_PCT  OREB  DREB  REB  AST  STL  BLK  TOV   PF   PTS  
0   0.833   1.0   7.3  8.3  8.7  1.8  1.3  5.3  3.3  25.0  

[1 rows x 24 columns]


In [ ]:

from datetime import datetime

def get_season():
    today = datetime.today()
    year = today.year
    month = today.month

    if month >= 10:  # October to December: new season starts
        start_year = year
    else:  # January to September: still part of the previous season
        start_year = year - 1

    current_season = f"{start_year}-{str(start_year + 1)[-2:]}"
    previous_season = f"{start_year - 1}-{str(start_year)[-2:]}"
    
    return current_season, previous_season

current_season, previous_season = get_season()
print("Current Season:", current_season)
print("Previous Season:", previous_season)
# TODO: need to find how many games played in current season so far to get the correct season

gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, season_type_all_star="Regular Season") # Game log

# boxscore = boxscoresummaryv2.BoxScoreSummaryV2(player_id=player_id) # Box score summary [Need for injury report] (TODO: Verify parameters)

# commoninfo = commonplayerinfo.CommonPlayerInfo(player_id=player_id) # Player info
# team_id = commoninfo.get_data_frames()[0]['TEAM_ID'].values[0]

